In [4]:
import torch

In [1]:
with open('names.txt', 'r', encoding='UTF-8') as f:
    words = f.read().splitlines()

In [22]:
# all characters in the dataset
chars = ['.'] + sorted(list(set(''.join(words))))
# string to integer encoder 
stoi = {ch:i for i, ch in enumerate(chars)}
# inverse mapping
itos = {i:ch for ch, i in stoi.items()}

In [15]:
# Create the training set of bigrams (x, y)
xs, ys = [], []
for w in words:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        idx1 = stoi[ch1]
        idx2 = stoi[ch2]
        xs.append(idx1)
        ys.append(idx2)

xs = torch.tensor(xs)
ys = torch.tensor(ys)
num = xs.nelement()
print('number of examples:', num)

# Check if MPS is available
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

# Initialize the network
g = torch.Generator(2147483647)
W = torch.randn((27, 27), generator=g, requires_grad=True, device=device)

number of examples: 228146


In [ ]:
# We can augment the loss function to incentivize model smoothing by
# encouraging the loss to bring all weights closer to 0.
# when all weights are 0, exp(0) -> 1 -> probs -> uniform.
# We can do this by tacking on an additional term (W**2).sum() which
# adds contribution the loss function the further a value is from 0
# We can scale this value with some constant (regularization strength) 
# to lower/raise its impact on the loss function
# This is exactly the same as adding a constant when model smoothing
# the counts matrix. 
# This is known as loss regularization
(W**2).mean()

tensor(2.1604, device='mps:0', grad_fn=<MeanBackward0>)

In [ ]:
import torch.nn.functional as F
# gradient descent
for k in range(100):

    # forward pass
    xenc = F.one_hot(xs, num_classes=27).float().to(device=device)
    logits = xenc @ W
    counts = logits.exp()
    probs = counts / counts.sum(1, keepdim=True) # make each row a prob distribution
    loss = -probs[torch.arange(num), ys].log().mean() + 0.01 * (W**2).mean()
    print(loss.item())

    # backward pass
    W.grad = None
    loss.backward()

    # Update gradient
    W.data += -50 * W.grad


2.469956636428833
2.4697906970977783
2.469628095626831
2.469468355178833
2.4693119525909424
2.4691579341888428
2.4690072536468506
2.4688591957092285
2.4687139987945557
2.468571186065674
2.468430995941162
2.4682934284210205
2.468158006668091
2.468024969100952
2.4678945541381836
2.467766284942627
2.4676403999328613
2.4675164222717285
2.4673943519592285
2.4672746658325195
2.4671571254730225
2.467041015625
2.4669270515441895
2.4668149948120117
2.4667046070098877
2.4665963649749756
2.466489553451538
2.466384172439575
2.466280937194824
2.466179132461548
2.466078758239746
2.465980291366577
2.465883255004883
2.465787410736084
2.4656929969787598
2.46560001373291
2.4655089378356934
2.465418577194214
2.465329885482788
2.4652421474456787
2.465156078338623
2.465070962905884
2.4649875164031982
2.46490478515625
2.4648232460021973
2.464743137359619
2.4646639823913574
2.464585781097412
2.4645087718963623
2.464432954788208
2.464357852935791
2.4642841815948486
2.4642112255096436
2.464139223098755
2.46406

In [28]:
# Sampling from the neural network
g = torch.Generator(2147483647)

for i in range(5):
    out = []
    ix = 0
    while True:
        # forward pass
        xenc = F.one_hot(torch.tensor([ix]), num_classes=27).float().to(device=device)
        logits = xenc @ W
        counts = logits.exp()
        probs = counts / counts.sum(1, keepdims=True)
        ix = torch.multinomial(probs, num_samples=1, replacement=True, generator=g).item()
        out.append(itos[ix])
        if itos[ix] == '.':
            out.pop()
            break
    print(''.join(out))

nintifon
bri
mensh
kisth
pt
